In [1]:
import sys
import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import polars as pl
import warnings
from typing import Tuple, Dict, Any

warnings.filterwarnings("ignore")


from mars.feature.binner import MarsOptimalBinner, MarsNativeBinner
from mars.utils.logger import set_log_level, logger

set_log_level("INFO")

def get_memory_usage() -> float:
    """获取当前进程内存占用 (MB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

print(f"✅ 环境就绪 | Polars: {pl.__version__} | Memory: {get_memory_usage():.2f} MB")

✅ 环境就绪 | Polars: 1.34.0 | Memory: 242.31 MB


In [2]:
def run_final_logic_tests():
    print("\n" + "="*60)
    print("🧪 模块 2: 最终版逻辑验证 (索引协议 & Join 路由)")
    print("="*60)

    # --- 1. 数值型：多特殊值与索引协议 ---
    print("\n[Case 1] 数值型：多特殊值索引验证")
    # 构造数据：含两个特殊值 -999, -998
    df_num = pl.DataFrame({"feat": [10, 20, -999, -998, None, 50]})
    y_num = [0, 0, 0, 0, 1, 1]
    
    binner = MarsOptimalBinner(
        features=["feat"], 
        n_bins=2, 
        special_values=[-999, -998],
        missing_values=[None]
    )
    binner.fit(df_num, y_num)
    
    # 验证 Index 模式
    res_idx = binner.transform(df_num, return_type="index")
    indices = res_idx["feat_bin"].to_list()
    print(f"   -> Indices: {indices}")
    # 协议验证：Missing=-1, Special1=-3, Special2=-4
    assert indices[4] == -1, "❌ Missing 索引错误"
    assert indices[2] == -3, "❌ 特殊值1 (-999) 索引错误"
    assert indices[3] == -4, "❌ 特殊值2 (-998) 索引错误"

    # --- 2. 类别型：高基数 Join 路由验证 ---
    print("\n[Case 2] 类别型：高基数 Join 路由测试")
    # 构造超过 join_threshold 的数据
    cats = [f"ID_{i}" for i in range(20)]
    df_cat = pl.DataFrame({"uid": cats})
    y_cat = np.random.randint(0, 2, 20)
    
    # 设置极低的阈值强制触发 Join
    binner_cat = MarsOptimalBinner(
        cat_features=["uid"], 
        n_bins=3, 
        join_threshold=5 
    )
    binner_cat.fit(df_cat, y_cat)
    
    # 验证 Join 后的索引输出
    res_cat = binner_cat.transform(df_cat, return_type="index")
    print(f"   -> Join Index Sample: {res_cat['uid_bin'].head(3).to_list()}")
    assert res_cat["uid_bin"].dtype in [pl.Int16, pl.Int8, pl.Int32], "❌ Join 路由输出类型错误"
    
    # 验证 Label 还原
    res_label = binner_cat.transform(df_cat, return_type="label")
    labels = res_label["uid_bin"].to_list()
    assert isinstance(labels[0], str) and "[" in labels[0], "❌ Join 路由 Label 还原错误"

    # --- 3. 类别型：Other 与 Special 瀑布流 ---
    print("\n[Case 3] 类别型：Other 与 Special 优先级")
    df_mix = pl.DataFrame({"city": ["A", "B", "C", "-999", None]})
    # 训练时只给 A, B
    df_train = pl.DataFrame({"city": ["A"]*10 + ["B"]*10})
    y_train = [1]*10 + [0]*10
    
    binner_mix = MarsOptimalBinner(
        cat_features=["city"], 
        special_values=["-999"]
    )
    binner_mix.fit(df_train, y_train)
    
    res_mix = binner_mix.transform(df_mix, return_type="index")
    mix_indices = res_mix["city_bin"].to_list()
    print(f"   -> Mix Indices: {mix_indices}")
    
    # 验证：A(0), B(1), C(Other:-2), -999(Special:-3), None(Missing:-1)
    assert mix_indices[2] == -2, "❌ 未见类别未归入 Other(-2)"
    assert mix_indices[3] == -3, "❌ 类别型特殊值映射错误"
    assert mix_indices[4] == -1, "❌ 类别型缺失值映射错误"

    # --- 4. 极致防御：类型转换稳定性 ---
    print("\n[Case 4] 类型防御：Int 列含空值")
    df_int = pl.DataFrame({"val": [1, 2, None]}, schema={"val": pl.Int64})
    binner_int = MarsOptimalBinner(features=["val"], n_bins=2)
    binner_int.fit(df_int, [0, 1, 0])
    # 验证不会因为 Int 列调用 is_nan 而崩溃
    try:
        binner_int.transform(df_int)
        print("   ✅ Int Null Safety 通关")
    except Exception as e:
        print(f"   ❌ Int Null Safety 失败: {e}")
        raise e
    
    # --- 5. 极致防御：混合类型配置抗毁性 ---
    print("\n[Case 5] 极致防御：混合类型配置抗毁性")
    print("   场景: 配置了字符串缺失值，但遇到了纯数值列。期望: 自动过滤，不报错。")
    
    df_safe = pl.DataFrame({"age": [20, 30, -999]}, schema={"age": pl.Int64})
    y_safe = [0, 1, 0]
    
    # 故意混入不兼容的类型 "unknown" 和 "N/A"
    binner_safe = MarsOptimalBinner(
        features=["age"], 
        n_bins=2, 
        missing_values=[-999, "unknown", "N/A"],
        special_values=[-1, "Error"]
    )
    
    try:
        binner_safe.fit(df_safe, y_safe)
        res_safe = binner_safe.transform(df_safe, return_type="index")
        
        # 验证 -999 是否正确被识别为 Missing (-1)
        # 如果 _get_safe_values 生效，"unknown" 应该被剔除，-999 保留
        idx_missing = res_safe.filter(pl.col("age") == -999)["age_bin"][0]
        assert idx_missing == -1, f"❌ 混合配置下数值缺失值未识别 (Got {idx_missing})"
        
        print("   ✅ Mixed Type Safety 通关")
    except Exception as e:
        print(f"   ❌ Mixed Type Safety 失败: {e}")
        raise e

    print("\n🎉 最终版逻辑验证全部通过！代码已达极致工业级水准。")

run_final_logic_tests()


🧪 模块 2: 最终版逻辑验证 (索引协议 & Join 路由)

[Case 1] 数值型：多特殊值索引验证
[MARS] 2026-01-21 00:10:07 - WARNING - ⚠️ 1 features failed during CART binning and fallbacked to single bin. Check `self.fit_failures_` for details. Sample fails: [('feat', 'Insufficient samples after cleaning.')]
[MARS] 2026-01-21 00:10:07 - WARNING - ⚠️ 1 numeric features are degenerate (single bin). Check `.fit_failures_` for details.
[MARS] 2026-01-21 00:10:07 - INFO - ⏱️ [MarsNativeBinner._fit_impl] finished in 0.0318s
[MARS] 2026-01-21 00:10:07 - INFO - ⏱️ [MarsOptimalBinner._fit_numerical_impl] finished in 0.0318s
   -> Indices: [0, 0, -3, -4, -1, 0]

[Case 2] 类别型：高基数 Join 路由测试
   -> Join Index Sample: [0, 0, 0]

[Case 3] 类别型：Other 与 Special 优先级
   -> Mix Indices: [0, 0, -2, -3, -1]

[Case 4] 类型防御：Int 列含空值
[MARS] 2026-01-21 00:10:12 - WARNING - ⚠️ 1 features failed during CART binning and fallbacked to single bin. Check `self.fit_failures_` for details. Sample fails: [('val', 'Insufficient samples after cleaning.')]
[MARS

In [3]:
import numpy as np
import polars as pl
from datetime import datetime

# ==========================================
# 1. 模拟全类型数据集 (5000行样本)
# ==========================================
n_rows = 5000
np.random.seed(42)

# 特征 A: 强正相关数值 (随着值增大，风险增高)
feat_pos = np.random.normal(0, 1, n_rows)
# 特征 B: 强负相关数值 (随着值增大，风险降低，测试 AUC 修正逻辑)
feat_neg = np.random.normal(0, 1, n_rows)
# 特征 C: 随机噪声 (IV/KS 应该接近 0)
feat_noise = np.random.normal(0, 1, n_rows)
# 特征 D: 含特殊值 (-999) 和 缺失值 (null) 的数值特征
feat_special = np.random.choice([10.0, 20.0, 30.0, -999.0, np.nan], n_rows, p=[0.3, 0.3, 0.2, 0.1, 0.1])
# 特征 E: 类别型特征 (A类高风险，B类低风险)
feat_cat = np.random.choice(['High_Risk', 'Low_Risk', 'Medium_Risk'], n_rows, p=[0.2, 0.5, 0.3])

# 构造目标变量 y (基于逻辑回归公式)
# 线性组合：pos越大y越趋向1，neg越大y越趋向0，cat='High_Risk'时y趋向1
logits = 1.5 * feat_pos - 2.0 * feat_neg + (np.where(feat_cat == 'High_Risk', 1.0, -1.0))
prob = 1 / (1 + np.exp(-logits))
y = (prob > np.random.rand(n_rows)).astype(int)

# 封装为 Polars DataFrame
df = pl.DataFrame({
    "num_pos": feat_pos,
    "num_neg": feat_neg,
    "num_noise": feat_noise,
    "num_special_missing": feat_special,
    "cat_risk": feat_cat,
    "target": y
})

print(f"✅ 测试数据构建完成，形状: {df.shape}")

# ==========================================
# 2. 初始化分箱器并拟合 (MarsOptimalBinner)
# ==========================================
# 我们定义 -999 为特殊值，None 会自动识别为缺失值
binner = MarsOptimalBinner(
    n_bins=5,
    special_values=[-999],
    missing_values=[np.nan],
    cat_features=["cat_risk"]
)

# 执行拟合
binner.fit(df, y=df["target"])

# ==========================================
# 3. 核心：产出分箱性能画像 (Indicator Profiling)
# ==========================================
# 我们开启 update_woe=True，这样计算后的权重会存入 binner 内部供转换使用
report = binner.profile_bin_performance(df, y=df["target"], update_woe=True)

# ==========================================
# 4. 结果查看与解读
# ==========================================
# 设置 Polars 显示选项，方便在 Jupyter 查看宽表
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(20)

print("\n" + "="*50)
print("📊 分箱指标计算全量报告 (按特征分组汇总)")
print("="*50)

# 我们对报告进行二次加工，方便一眼看出哪些特征强，哪些特征弱
summary = (
    report.group_by("feature")
    .agg([
        pl.col("IV").first(),
        pl.col("KS").first(),
        pl.col("AUC").first()
    ])
    .sort("IV", descending=True)
)
print(summary)

print("\n" + "="*50)
print("🔍 详细分箱明细 (展示正相关、负相关、特殊值处理效果)")
print("="*50)

# 查看特征 num_pos (正相关) 和 num_special_missing (含特殊值) 的明细
target_show = ["num_pos_bin", "num_neg_bin", "num_special_missing_bin", "cat_risk_bin"]
display_detail = report.filter(pl.col("feature").is_in(target_show))
display(display_detail)

# ==========================================
# 5. 验证持久化逻辑 (Persistence Test)
# ==========================================
print("\n" + "="*50)
print("💾 持久化测试: to_dict -> from_dict -> transform")
print("="*50)

# 导出字典
binner_dict = binner.to_dict()

# 从字典恢复新实例 (模拟推理环境)
new_binner = MarsOptimalBinner.from_dict(binner_dict)

# 在新实例上进行转换测试
test_X = df.head(5)
# 转换出 WOE 值
woe_res = new_binner.transform(test_X, return_type="woe")

print("▶️ 推理实例转换结果 (WOE 后缀列):")
display(woe_res.select(pl.col("^.*_woe$")))

# 检查转换出来的 WOE 是否在之前 report 的明细里能对上
sample_feat = "num_pos"
sample_woe = woe_res[0, f"{sample_feat}_woe"]
print(f"✅ 随机验证: 特征 {sample_feat} 的首行 WOE 值为 {sample_woe:.4f}")

✅ 测试数据构建完成，形状: (5000, 6)
[MARS] 2026-01-21 00:10:12 - INFO - ⏱️ [MarsNativeBinner._fit_impl] finished in 0.0216s
[MARS] 2026-01-21 00:10:14 - INFO - ⏱️ [MarsOptimalBinner._fit_numerical_impl] finished in 2.1514s
[MARS] 2026-01-21 00:10:15 - INFO - ⏱️ [MarsOptimalBinner.profile_bin_performance] finished in 0.0208s

📊 分箱指标计算全量报告 (按特征分组汇总)
shape: (5, 4)
┌─────────────────────────┬──────────┬──────────┬──────────┐
│ feature                 ┆ IV       ┆ KS       ┆ AUC      │
│ ---                     ┆ ---      ┆ ---      ┆ ---      │
│ str                     ┆ f64      ┆ f64      ┆ f64      │
╞═════════════════════════╪══════════╪══════════╪══════════╡
│ num_neg_bin             ┆ 1.34238  ┆ 0.411279 ┆ 0.788264 │
│ num_pos_bin             ┆ 0.691754 ┆ 0.322101 ┆ 0.714682 │
│ cat_risk_bin            ┆ 0.166764 ┆ 0.162455 ┆ 0.589463 │
│ num_noise_bin           ┆ 0.004329 ┆ 0.030339 ┆ 0.515225 │
│ num_special_missing_bin ┆ 0.000645 ┆ 0.011587 ┆ 0.506362 │
└─────────────────────────┴──────────

feature,bin_label,count,bad,good,count_dist,bad_rate,bad_dist,good_dist,woe,bin_iv,cum_bad_dist,cum_good_dist,bin_ks,IV,KS,AUC,Lift
str,str,u32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""num_neg_bin""","""00_[-inf, -1.32)""",498,437,61,0.0996,0.87751,0.206229,0.021173,2.27621,0.421227,1.0,1.0,1.2482e-10,1.34238,0.411279,0.788264,2.070576
"""num_neg_bin""","""01_[-1.32, -0.481)""",1092,739,353,0.2184,0.67674,0.348749,0.122527,1.046018,0.236633,0.793771,0.978827,0.185056,1.34238,0.411279,0.788264,1.596838
"""num_neg_bin""","""02_[-0.481, 0.321)""",1574,665,909,0.3148,0.42249,0.313827,0.315515,-0.005365,0.000009,0.445021,0.8563,0.411279,1.34238,0.411279,0.788264,0.99691
"""num_neg_bin""","""03_[0.321, 0.891)""",896,199,697,0.1792,0.222098,0.093912,0.24193,-0.946281,0.140066,0.131194,0.540784,0.40959,1.34238,0.411279,0.788264,0.524064
"""num_neg_bin""","""04_[0.891, inf)""",940,79,861,0.188,0.084043,0.037282,0.298855,-2.08143,0.544446,0.037282,0.298855,0.261573,1.34238,0.411279,0.788264,0.198307
"""num_special_missing_bin""","""00_[-inf, 15)""",1429,611,818,0.2858,0.427572,0.288344,0.283929,0.015428,0.000068,0.900425,0.903506,0.003081,0.000645,0.011587,0.506362,1.0089
"""num_special_missing_bin""","""01_[15, 25)""",1496,639,857,0.2992,0.427139,0.301557,0.297466,0.01366,0.000056,0.612081,0.619577,0.007495,0.000645,0.011587,0.506362,1.007879
"""num_special_missing_bin""","""02_[25, inf)""",1039,431,608,0.2078,0.414822,0.203398,0.211038,-0.036873,0.000282,0.203398,0.211038,0.00764,0.000645,0.011587,0.506362,0.978815
"""num_special_missing_bin""","""Special_-999""",489,211,278,0.0978,0.431493,0.099575,0.096494,0.03143,0.000097,1.0,1.0,1.2482e-10,0.000645,0.011587,0.506362,1.018152



💾 持久化测试: to_dict -> from_dict -> transform
▶️ 推理实例转换结果 (WOE 后缀列):


num_neg_woe,num_special_missing_woe,num_noise_woe,num_pos_woe,cat_risk_woe
f64,f64,f64,f64,f64
-0.005365,0.01366,0.043804,0.712656,-0.248924
-0.005365,0.015428,0.043804,-0.074826,-0.248924
2.27621,0.03143,0.043804,0.712656,0.804199
-0.005365,0.01366,0.043804,0.712656,-0.131933
-0.946281,0.03143,-0.100258,-0.074826,0.804199


✅ 随机验证: 特征 num_pos 的首行 WOE 值为 0.7127


In [6]:
(
    report
    .group_by("feature")
    .sum()
)

feature,bin_label,count,bad,good,count_dist,bad_rate,bad_dist,good_dist,lift,woe,bin_iv,cum_bad_dist,cum_good_dist,bin_ks,IV,KS,AUC
str,str,u32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""num_noise_bin""",null,5000,2119,2881,1.0,1.235438,1.0,1.0,2.915144,-0.148961,0.004329,1.529495,1.584866,0.055371,0.012988,0.091016,1.545674
"""num_neg_bin""",null,5000,2119,2881,1.0,2.282881,1.0,1.0,5.386695,0.289152,1.34238,2.407268,3.674766,1.267498,6.711902,2.056393,3.941322
"""cat_risk_bin""",null,5000,2119,2881,1.0,1.378152,1.0,1.0,3.251892,0.423342,0.166764,2.150071,2.438042,0.287972,0.500291,0.487365,1.768389
"""num_special_missing_bin""",null,5000,2119,2881,1.0,2.116016,1.0,1.0,4.99296,-0.012534,0.000645,3.026428,3.05623,0.029803,0.003227,0.057933,2.531811
"""num_pos_bin""",null,5000,2119,2881,1.0,2.200706,1.0,1.0,5.192795,0.00873,0.691754,2.534686,3.382853,0.848167,3.458768,1.610504,3.573409


In [7]:
import polars as pl

def test_categorical_trap():
    print("🔥 Polars 'Categorical Trap' 现场还原实验 🔥\n")

    # 1. 构造简单数据
    df = pl.DataFrame({"age": [18, 55, 18, 55]})
    
    # 2. 执行分箱 (Cut)
    # 关键点：我们将标签设为 "100" 和 "200"，而不是默认的 "0", "1"
    # 这样如果结果出现 0 或 1，你就知道它在“读编号”而不是“读内容”了
    breaks = [30.0]
    labels = ["100", "200"]  # 假设这是我们的业务映射值
    
    df_cut = df.with_columns(
        pl.col("age").cut(breaks, labels=labels, left_closed=True).alias("cat_col")
    )
    
    print("--- 1. 原始分箱结果 (Categorical 类型) ---")
    print(df_cut)
    
    # 3. ❌ 错误做法：直接转 Int
    # 这相当于偷看了 Polars 的“内部账本”
    df_wrong = df_cut.select(
        pl.col("cat_col"),
        pl.col("cat_col").cast(pl.Int16).alias("physical_index_wrong")
    )
    
    print("\n--- 2. ❌ 错误做法 (.cast(pl.Int16)) ---")
    print("现象：你看到的是 0 和 1，而不是我们定义的 100 和 200。")
    print("原因：这是 Polars 内部分配的存储 ID (Physical Index)。")
    print("后果：如果你的 mapping 字典是 {100: 'Young', 200: 'Old'}，用 0 和 1 去查肯定报错！")
    print(df_wrong)

    # 4. ✅ 正确做法：先转 String 再转 Int
    # 这相当于强制 Polars 把“内容”吐出来
    df_right = df_cut.select(
        pl.col("cat_col"),
        pl.col("cat_col").cast(pl.Utf8).cast(pl.Int16).alias("logical_value_right")
    )
    
    print("\n--- 3. ✅ 正确做法 (.cast(pl.Utf8).cast(pl.Int16)) ---")
    print("现象：成功拿到了 100 和 200。")
    print("原因：先转字符串提取了'标签文本'，再转数字得到了'业务数值'。")
    print(df_right)

    # 5. 💀 模拟“索引偏移” (为什么你会看到 3 ?)
    # Polars 的物理索引是从 0 开始排的。
    # 如果你的数据里 labels=["0", "1", "2"]
    # 正常情况下： "0"->0, "1"->1, "2"->2 (看起来没问题)
    # 
    # 但如果 Polars 内部决定保留 0 给 Null 值 (或某种特殊对其)，它可能会变成：
    # Null -> 0
    # "0"  -> 1
    # "1"  -> 2
    # "2"  -> 3  <-- 💥 你的 mapping 里只有 2，但它给了你 3！
    
    print("\n--- 4. 💀 为什么之前你的代码会出现 '3' ? ---")
    print("当数据变复杂时，Polars 可能会把物理索引向后推移 (Off-by-one)。")
    print("比如标签 '2' 的物理索引变成了 3。")
    print("如果你用旧代码直接取 ID (拿到3)，去查只有 {0,1,2} 的字典，自然就匹配失败，原来的 3 就直接显示出来了。")
    print("而用新代码，无论它内部排到几号，我们读出来的永远是字符串 '2'。")

if __name__ == "__main__":
    test_categorical_trap()

🔥 Polars 'Categorical Trap' 现场还原实验 🔥

--- 1. 原始分箱结果 (Categorical 类型) ---
shape: (4, 2)
┌─────┬─────────┐
│ age ┆ cat_col │
│ --- ┆ ---     │
│ i64 ┆ cat     │
╞═════╪═════════╡
│ 18  ┆ 100     │
│ 55  ┆ 200     │
│ 18  ┆ 100     │
│ 55  ┆ 200     │
└─────┴─────────┘

--- 2. ❌ 错误做法 (.cast(pl.Int16)) ---
现象：你看到的是 0 和 1，而不是我们定义的 100 和 200。
原因：这是 Polars 内部分配的存储 ID (Physical Index)。
后果：如果你的 mapping 字典是 {100: 'Young', 200: 'Old'}，用 0 和 1 去查肯定报错！
shape: (4, 2)
┌─────────┬──────────────────────┐
│ cat_col ┆ physical_index_wrong │
│ ---     ┆ ---                  │
│ cat     ┆ i16                  │
╞═════════╪══════════════════════╡
│ 100     ┆ 0                    │
│ 200     ┆ 1                    │
│ 100     ┆ 0                    │
│ 200     ┆ 1                    │
└─────────┴──────────────────────┘

--- 3. ✅ 正确做法 (.cast(pl.Utf8).cast(pl.Int16)) ---
现象：成功拿到了 100 和 200。
原因：先转字符串提取了'标签文本'，再转数字得到了'业务数值'。
shape: (4, 2)
┌─────────┬─────────────────────┐
│ cat_col ┆ logical_value_right │
│ --- 

In [5]:
# ==========================================
# 模块 3: 大规模压力测试 (20万行 x 1000列 - 三模式对比)
# ==========================================

N_ROWS = 200_000
N_COLS = 1000

def run_stress_test():
    print("\n" + "="*80)
    print(f"🔥 模块 3: 大规模压力测试 ({N_ROWS:,}行 x {N_COLS}列)")
    print("="*80)
    
    # 1. 生成海量数据 (Float32 节省内存)
    print(f"🚀 [DataGen] 正在生成数据...")
    data_matrix = np.random.randn(N_ROWS, N_COLS).astype(np.float32)
    y = (data_matrix[:, 0] > 0).astype(int)
    df = pl.from_numpy(data_matrix, schema=[f"f_{i}" for i in range(N_COLS)])
    print(f"✅ 数据准备完毕 | 内存占用: {get_memory_usage():.2f} MB")

    results = []
    methods = [
        # "quantile", 
        # "uniform",
        "cart"
        ]

    for method in methods:
        print(f"\n🧪 Testing Pre-binning Method: [{method.upper()}]")
        print("-" * 45)
        
        # 清理内存
        gc.collect()
        time.sleep(1) # 给系统回收缓冲的时间
        mem_start = get_memory_usage()
        t_start = time.time()
        
        # 2. 训练 MarsOptimalBinner
        # 混合动力引擎：Stage 1 (method) -> Stage 2 (CP Solver)
        binner = MarsOptimalBinner(
            n_bins=5, 
            n_prebins=50, 
            prebinning_method=method,
            n_jobs=-1,      
            time_limit=2    
        )
        
        binner.fit(df, y)
        t_fit = time.time() - t_start
        
        # 3. 转换 (默认返回 index 模式以测试极致性能)
        t_trans_start = time.time()
        res = binner.transform(df, return_type="index")
        # 强制触发 Polars 计算
        _ = res.select(pl.col("f_0_bin")).height 
        t_trans = time.time() - t_trans_start
        
        mem_peak = get_memory_usage() - mem_start
        
        results.append({
            "Method": method,
            "Fit Time(s)": t_fit,
            "Trans Time(s)": t_trans,
            "Mem Gain(MB)": mem_peak,
            "Throughput(f/s)": N_COLS / t_fit
        })
        
        print(f"   ⏱️  Fit Time:       {t_fit:.2f} s")
        print(f"   🚀  Trans Time:     {t_trans:.2f} s")
        print(f"   💾  Mem Overhead:   {mem_peak:.2f} MB")

    # 4. 打印汇总看板
    print("\n" + "🏆 STRESS TEST SUMMARY".center(60, "="))
    report_df = pd.DataFrame(results)
    display(report_df.round(3))
    
    # 彻底清理
    del df, data_matrix; gc.collect()

run_stress_test()


🔥 模块 3: 大规模压力测试 (200,000行 x 1000列)
🚀 [DataGen] 正在生成数据...
✅ 数据准备完毕 | 内存占用: 1807.44 MB

🧪 Testing Pre-binning Method: [CART]
---------------------------------------------
[MARS] 2026-01-20 23:55:59 - INFO - ⏱️ [MarsNativeBinner._fit_impl] finished in 14.6041s
[MARS] 2026-01-20 23:56:10 - INFO - ⏱️ [MarsOptimalBinner._fit_numerical_impl] finished in 25.5990s
[MARS] 2026-01-20 23:56:10 - WARNING - ⚠️ MarsOptimalBinner: 3 features encountered issues (3 num, 0 cat). Fallback applied. Check `.fit_failures_` for details. Sample: [('f_0', 'Solver collapsed to single bin; fallback to pre-bins'), ('f_153', 'Solver collapsed to single bin; fallback to pre-bins')]
   ⏱️  Fit Time:       25.63 s
   🚀  Trans Time:     0.81 s
   💾  Mem Overhead:   592.86 MB

===================🏆 STRESS TEST SUMMARY====================


,Method,Fit Time(s),Trans Time(s),Mem Gain(MB),Throughput(f/s)
0,cart,25.632,0.809,592.859,39.013


In [ ]:
import sys
import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import polars as pl
import warnings
from typing import Tuple, Dict, Any

warnings.filterwarnings("ignore")


from mars.feature.binner import MarsOptimalBinner, MarsNativeBinner
from mars.utils.logger import set_log_level, logger

set_log_level("INFO")

def get_memory_usage() -> float:
    """获取当前进程内存占用 (MB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

print(f"✅ 环境就绪 | Polars: {pl.__version__} | Memory: {get_memory_usage():.2f} MB")
from optbinning import BinningProcess

def benchmark_optimal_binning_with_check():
    # ==========================================
    # 1. 数据生成 (模拟大宽表场景)
    # ==========================================
    N_ROWS = 200_000
    N_COLS = 10
    
    print(f"🛠️ 正在生成测试数据: {N_ROWS} 行 x {N_COLS} 列...")
    np.random.seed(42)
    X = np.random.randn(N_ROWS, N_COLS).astype(np.float32)
    
    # 构造 Target: 简单的线性关系 + 噪声
    # 这种构造方式保证了特征具有一定的单调性趋势，利于 OptimalBinning 求解
    y = (X[:, 0] * 5 + X[:, 1] * 2 + np.random.randn(N_ROWS) > 0).astype(int)
    
    feat_names = [f"f_{i}" for i in range(N_COLS)]
    
    # 准备两种格式的数据
    df_pl = pl.from_numpy(X, schema=feat_names) # 给 Mars 用
    df_pd = pd.DataFrame(X, columns=feat_names) # 给 OptBinning 用
    
    print("✅ 数据准备完毕。\n")

    # ==========================================
    # 2. 定义测试配置
    # ==========================================
    methods = [
        "quantile", 
        "uniform", 
        "cart"
        ]
    
    print(f"{'Method':<10} | {'Engine':<12} | {'Fit (s)':<10} | {'Trans (s)':<10} | {'Total (s)':<10} | {'Speedup':<8}")
    print("-" * 80)

    for method in methods:
        # ==========================================
        # Round 1: Mars (Hybrid Engine)
        # ==========================================
        gc.collect()
        t0 = time.time()
        
        mars_model = MarsOptimalBinner(
            n_bins=5, 
            n_prebins=50,       
            prebinning_method=method, 
            n_jobs=-1,
            time_limit=2        
        )
        
        mars_model.fit(df_pd, y)
        t_fit_end = time.time()
        
        res = mars_model.transform(df_pd)
        # _ = res.select(pl.col(f"f_0_bin")).to_series() 
        print(type(res))
        t_trans_end = time.time()
        
        mars_fit_time = t_fit_end - t0
        mars_trans_time = t_trans_end - t_fit_end
        mars_total = mars_fit_time + mars_trans_time

        print(f"{method:<10} | {'Mars':<12} | {mars_fit_time:<10.4f} | {mars_trans_time:<10.4f} | {mars_total:<10.4f} | {'1.0x':<8}")

        # ==========================================
        # Round 2: OptBinning (Standard)
        # ==========================================
        gc.collect()
        t0 = time.time()
        
        # 1. 构造详细的参数配置字典
        fit_params = {}
        for col in feat_names:
            fit_params[col] = {
                # --- 预分箱参数 ---
                'prebinning_method': method, 
                'max_n_prebins': 50,
                
                # --- [关键] 核心优化参数 (对齐 Mars) ---
                # 强制 OptBinning 也进行双向搜索 (升序+降序)，而不是默认的猜方向
                'monotonic_trend': 'auto_asc_desc',  
                
                # 最小箱占比 (虽然后面 BinningProcess 也有这个参数，但写在这里优先级最高)
                'min_bin_size': 0.05
            }

        # 2. 初始化 BinningProcess
        opt_process = BinningProcess(
            variable_names=feat_names,
            max_n_bins=5,  
            n_jobs=-1,  
            binning_fit_params=fit_params 
        )
        
        opt_process.fit(df_pd, y)
        t_fit_end = time.time()
        
        _ = opt_process.transform(df_pd, metric="indices")
        t_trans_end = time.time()
        
        opt_fit_time = t_fit_end - t0
        opt_trans_time = t_trans_end - t_fit_end
        opt_total = opt_fit_time + opt_trans_time
        
        speedup = opt_total / mars_total
        print(f"{method:<10} | {'OptBinning':<12} | {opt_fit_time:<10.4f} | {opt_trans_time:<10.4f} | {opt_total:<10.4f} | {speedup:<8.1f}x")

        # ==========================================
        # 🔍 切点一致性抽检 (Check Splits) 冒烟测试 
        # ==========================================
        print(f"   [🔍 切点抽样对比: {method}]")
        
        # 随机抽取 3 个特征
        check_cols = np.random.choice(feat_names, 3, replace=False)
        
        for col in check_cols:
            # 1. 获取 Mars 切点 (去除 -inf, inf)
            if col in mars_model.bin_cuts_:
                mars_cuts = mars_model.bin_cuts_[col]
                # Mars 的格式是 [-inf, c1, c2, ..., inf]，比较时去掉无穷大
                mars_splits = [c for c in mars_cuts if c != float('-inf') and c != float('inf')]
            else:
                mars_splits = []

            # 2. 获取 OptBinning 切点
            # OptBinning 对象可能没有生成 splits (比如单调性约束无法满足)，需要处理异常
            try:
                opt_bin_obj = opt_process.get_binned_variable(col)
                if opt_bin_obj is not None:
                    opt_splits = opt_bin_obj.splits.tolist()
                else:
                    opt_splits = []
            except Exception:
                opt_splits = []
            
            # 3. 打印对比
            # 保留4位小数以便观察
            m_str = ", ".join([f"{x:.4f}" for x in mars_splits])
            o_str = ", ".join([f"{x:.4f}" for x in opt_splits])
            
            # 简单判断是否完全一致 (允许极小误差)
            is_close = False
            if len(mars_splits) == len(opt_splits):
                if len(mars_splits) == 0:
                    is_close = True
                else:
                    is_close = np.allclose(mars_splits, opt_splits, atol=1e-4)
            
            status_icon = "✅" if is_close else "⚠️"
            
            print(f"   - Feature {col:<5}: {status_icon}")
            print(f"     Mars: [{m_str}]")
            print(f"     Opt : [{o_str}]")
        
        print("-" * 80)

if __name__ == "__main__":
    benchmark_optimal_binning_with_check()

✅ 环境就绪 | Polars: 1.34.0 | Memory: 228.00 MB
🛠️ 正在生成测试数据: 200000 行 x 10 列...
✅ 数据准备完毕。

Method     | Engine       | Fit (s)    | Trans (s)  | Total (s)  | Speedup 
--------------------------------------------------------------------------------
[MARS] 2026-01-20 23:55:17 - INFO - ⏱️ [MarsNativeBinner._fit_impl] finished in 0.0184s
[MARS] 2026-01-20 23:55:22 - INFO - ⏱️ [MarsOptimalBinner._fit_numerical_impl] finished in 5.2761s
<class 'pandas.core.frame.DataFrame'>
quantile   | Mars         | 5.2911     | 0.0362     | 5.3272     | 1.0x    
